# 01 — FBref Data Cleaning

**Goal:** Load the 35 CSVs produced by `scrape_fbref.py`, standardize columns, filter players under 900 minutes, convert counting stats to per-90, then merge all 5 stat tables into one analysis-ready file.

**Input:** `data/raw/fbref/<league>_<stat_type>.csv` (35 files)  
**Output:** `data/cleaned/fbref_cleaned.csv`

**File naming convention from scraper:**  
`{league_slug}_{stat_type}.csv` e.g. `ned_eredivisie_standard.csv`

**Stat types:** `standard`, `passing`, `possession`, `defense`, `misc`

## 1. Imports and Config

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR    = Path('../data/raw/fbref')
CLEANED_DIR = Path('../data/cleaned')
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

MIN_MINUTES = 900

# The 5 stat types the scraper pulls
STAT_TYPES = ['standard', 'passing', 'possession', 'defense', 'misc']

# Merge keys — soccerdata saves these as index columns
MERGE_KEYS = ['league', 'season', 'team', 'player']

## 2. Load and Inspect Raw Files

soccerdata saves with `index=True`, so the MultiIndex (league, season, team, player) becomes the first columns in the CSV. We read with `index_col` unset and call `reset_index()` to keep everything flat.

In [7]:
def load_stat_type(stat_type: str, raw_dir: Path) -> pd.DataFrame:
    """
    Load all CSVs for a given stat type, combine across leagues,
    and return a single flat DataFrame.
    """
    files = sorted(raw_dir.glob(f'*_{stat_type}.csv'))
    if not files:
        print(f'  [WARN] No files found for stat type: {stat_type}')
        return pd.DataFrame()

    frames = []
    for f in files:
        df = pd.read_csv(f)
        # soccerdata MultiIndex lands as the first N columns — reset ensures
        # league/season/team/player are plain columns regardless of how it was saved
        frames.append(df)
        print(f'  {f.name}: {len(df)} rows, {df.shape[1]} cols')

    combined = pd.concat(frames, ignore_index=True)
    print(f'  -> Combined {stat_type}: {combined.shape}')
    return combined


print('Loading raw FBref files...')
raw = {}
for st in STAT_TYPES:
    print(f'\n[{st}]')
    raw[st] = load_stat_type(st, RAW_DIR)

print('\nDone loading.')

Loading raw FBref files...

[standard]
  [WARN] No files found for stat type: standard

[passing]
  [WARN] No files found for stat type: passing

[possession]
  [WARN] No files found for stat type: possession

[defense]
  [WARN] No files found for stat type: defense

[misc]
  [WARN] No files found for stat type: misc

Done loading.


## 3. Inspect Column Names

Run this cell to see what soccerdata actually named the columns — use the output to verify/extend the rename map in the next cell.

In [3]:
for st, df in raw.items():
    if df.empty:
        continue
    print(f'\n[{st}] columns:')
    print(list(df.columns))

## 4. Standardize Column Names

In [4]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase, strip, replace non-alphanumeric with underscores."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )
    return df


# Rename map — covers soccerdata FBref column names.
# After running cell 3, add any additional renames here.
COLUMN_RENAME = {
    # Identity
    'player':           'player_name',
    'squad':            'club',
    'team':             'club',
    'comp':             'league',
    'pos':              'position',
    'age':              'age',
    # Standard
    'mp':               'matches_played',
    'min':              'minutes',
    'minutes':          'minutes',
    'gls':              'goals',
    'ast':              'assists',
    'pk':               'penalties_scored',
    'pkatt':            'penalties_attempted',
    'xg':               'xg',
    'npxg':             'npxg',
    'xag':              'xag',
    'prgc':             'progressive_carries',
    'prgp':             'progressive_passes',
    'prgr':             'progressive_receptions',
    # Passing
    'cmp_percent':      'pass_cmp_pct',
    '1_3':              'passes_into_final_third',
    'ppa':              'passes_into_penalty_area',
    'tb':               'through_balls',
    'kp':               'key_passes',
    # Possession
    'att_3rd':          'touches_att_third',
    'succ':             'dribbles_completed',
    'succ_percent':     'dribble_success_pct',
    'carries_1_3':      'carries_into_final_third',
    'cpa':              'carries_into_penalty_area',
    # Defense
    'tkl':              'tackles',
    'tklw':             'tackles_won',
    'int':              'interceptions',
    'tkl_int':          'tackles_plus_interceptions',
    'press':            'pressures',
    'succ_press':       'pressure_regains',
    'recov':            'ball_recoveries',
    # Misc
    'won_percent':      'aerial_win_pct',
    'won':              'aerial_won',
    'lost':             'aerial_lost',
}

raw_std = {}
for st, df in raw.items():
    if df.empty:
        raw_std[st] = df
        continue
    df = standardize_columns(df)
    df = df.rename(columns={k: v for k, v in COLUMN_RENAME.items() if k in df.columns})
    raw_std[st] = df
    print(f'[{st}] renamed columns: {list(df.columns)}')

## 5. Identify Merge Keys

Determine which identity columns are present so we can merge tables correctly.

In [5]:
# Candidate merge key columns (after renaming)
CANDIDATE_KEYS = ['player_name', 'club', 'league', 'season']

# Find keys that actually exist across all non-empty tables
available_keys = None
for st, df in raw_std.items():
    if df.empty:
        continue
    present = [k for k in CANDIDATE_KEYS if k in df.columns]
    if available_keys is None:
        available_keys = set(present)
    else:
        available_keys &= set(present)

MERGE_ON = list(available_keys) if available_keys else ['player_name']
print(f'Merging on: {MERGE_ON}')

Merging on: ['player_name']


## 6. Filter — Outfield Players and Minimum Minutes

Apply the minutes filter on the standard table first, then use the surviving players to filter all other tables.

In [6]:
df_standard = raw_std.get('standard', pd.DataFrame())

if df_standard.empty:
    raise ValueError('Standard stats table is empty — check scraper output.')

# Minutes column — soccerdata may name it 'minutes' or '90s' (= minutes / 90)
# Check which is present and create a unified 'minutes' column
if 'minutes' not in df_standard.columns and '90s' in df_standard.columns:
    df_standard['minutes'] = pd.to_numeric(df_standard['90s'], errors='coerce') * 90
    print('Derived minutes from 90s column')
else:
    df_standard['minutes'] = pd.to_numeric(df_standard['minutes'], errors='coerce')

# Drop goalkeepers
if 'position' in df_standard.columns:
    before = len(df_standard)
    df_standard = df_standard[~df_standard['position'].str.contains('GK', na=False)]
    print(f'Removed {before - len(df_standard)} GKs')

# Apply minutes filter
before = len(df_standard)
df_standard = df_standard[df_standard['minutes'] >= MIN_MINUTES].copy()
print(f'After {MIN_MINUTES}+ min filter: {len(df_standard)} players (removed {before - len(df_standard)})')

df_standard.head(3)

ValueError: Standard stats table is empty — check scraper output.

## 7. Merge All 5 Stat Tables

In [ ]:
df_merged = df_standard.copy()

for st in ['passing', 'possession', 'defense', 'misc']:
    df_other = raw_std.get(st, pd.DataFrame())
    if df_other.empty:
        print(f'[SKIP] {st} table is empty')
        continue

    # Only keep columns that don't already exist (avoid duplicate identity cols)
    existing_cols = set(df_merged.columns)
    cols_to_add = MERGE_ON + [c for c in df_other.columns if c not in existing_cols]
    df_other_slim = df_other[cols_to_add].drop_duplicates(subset=MERGE_ON)

    df_merged = df_merged.merge(df_other_slim, on=MERGE_ON, how='left')
    print(f'After merging {st}: {df_merged.shape}')

print(f'\nFinal merged shape: {df_merged.shape}')
df_merged.head(3)

## 8. Convert Counting Stats to Per-90 Values

In [ ]:
# All counting stats across all 5 tables that should be per-90 normalized
COUNTING_STATS = [
    # Standard
    'goals', 'assists', 'xg', 'npxg', 'xag',
    'progressive_carries', 'progressive_passes', 'progressive_receptions',
    # Passing
    'passes_into_final_third', 'passes_into_penalty_area', 'through_balls', 'key_passes',
    # Possession
    'touches_att_third', 'dribbles_completed',
    'carries_into_final_third', 'carries_into_penalty_area',
    # Defense
    'tackles', 'tackles_won', 'interceptions', 'tackles_plus_interceptions',
    'pressures', 'pressure_regains', 'ball_recoveries',
    # Misc
    'aerial_won', 'aerial_lost',
]


def add_per90(df: pd.DataFrame, stats: list, minutes_col: str = 'minutes') -> pd.DataFrame:
    df = df.copy()
    minutes = pd.to_numeric(df[minutes_col], errors='coerce').replace(0, np.nan)
    for stat in stats:
        if stat in df.columns:
            df[stat] = pd.to_numeric(df[stat], errors='coerce')
            df[f'{stat}_p90'] = (df[stat] / minutes * 90).round(3)
    return df


df_p90 = add_per90(df_merged, COUNTING_STATS)
p90_cols = [c for c in df_p90.columns if c.endswith('_p90')]
print(f'Added {len(p90_cols)} per-90 columns')
df_p90[['player_name', 'club', 'league', 'minutes'] + p90_cols[:6]].head()

## 9. Derived Metrics

Ground duel win % is not a raw FBref column — construct it from tackles won and interceptions.

In [ ]:
# Ground duel win % proxy: tackles won + interceptions / total tackles attempted + interceptions
# Documented as a constructed metric (not a raw column)
if all(c in df_p90.columns for c in ['tackles_won', 'tackles', 'interceptions']):
    denom = df_p90['tackles'] + df_p90['interceptions']
    numer = df_p90['tackles_won'] + df_p90['interceptions']
    df_p90['ground_duel_win_pct'] = (numer / denom.replace(0, np.nan) * 100).round(1)
    print('Created ground_duel_win_pct')

# Press success rate
if all(c in df_p90.columns for c in ['pressure_regains', 'pressures']):
    df_p90['pressure_success_pct'] = (
        df_p90['pressure_regains'] / df_p90['pressures'].replace(0, np.nan) * 100
    ).round(1)
    print('Created pressure_success_pct')

## 10. Sanity Check

In [ ]:
print('Final shape:', df_p90.shape)
print('\nMissing values (top 15):')
print(df_p90.isnull().sum().sort_values(ascending=False).head(15))
print('\nLeague distribution:')
if 'league' in df_p90.columns:
    print(df_p90['league'].value_counts())
print('\nPosition distribution:')
if 'position' in df_p90.columns:
    print(df_p90['position'].value_counts())

## 11. Save Cleaned Data

In [ ]:
output_path = CLEANED_DIR / 'fbref_cleaned.csv'
df_p90.to_csv(output_path, index=False)
print(f'Saved {len(df_p90)} rows, {df_p90.shape[1]} columns to {output_path}')